# Content-Based и Temporal Content-Based

Здесь сравниваю обычную Content-Based модель и вариант с учетом последних оценок пользователя.

Сначала готовим данные и считаем рекомендации. Потом проверяем несколько значений `gamma` на Validation, выбираем лучшее и оцениваем обе модели на Test. В конце сравниваем результаты с Transformer и Popularity.

Метрики берем из `src/evaluation/metrics.py`, поэтому одни и те же формулы используются для всех моделей.

Поскольку мы используем только жанры фильмов, модель получает довольно ограниченную информацию об их содержании и предпочтениях пользователей. Поэтому ожидать высокой точности рекомендаций от такого подхода не стоит.


## 1. Импорты

Подключаем библиотеки и функции из нашего проекта. Путь к корню проекта определяем автоматически, чтобы ноутбук можно было запускать и из `notebooks/`.


In [81]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm import tqdm

# Находим корень проекта: папку, в которой находится src/.
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Не найдена папка src/. Открой проект transformer-recommender.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.load_data import load_movielens
from src.data.preprocess import sort_interactions
import src.models.content_based as cb
import src.evaluation.metrics as metrics

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "ml-1m"
K = 10

## 2. Загружаем данные и готовим жанры

В `ratings` хранятся оценки пользователей, а в `movies` названия и жанры фильмов. Для совместимости с Transformer используем те же 3706 фильмов и индексы `movie_idx`.

`genre_matrix` содержит строки для фильмов и столбцы для жанров. Значение 1 означает, что у фильма есть этот жанр.


In [82]:
ratings, movies = load_movielens(DATA_DIR)

movie_ids = sorted(ratings["movie_id"].unique())
movie2idx = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}
num_movies = len(movie_ids)

# Строка матрицы = фильм, столбец = жанр, значение 0 или 1.
genre_matrix, genre_names = cb.build_genre_matrix(
    movies=movies,
    movie_ids=movie_ids,
)

print("Оценок:", len(ratings))
print("Фильмов:", num_movies)
print("Размерность матрицы жанров:", genre_matrix.shape)

display(pd.DataFrame(
    genre_matrix[:5],
    columns=genre_names,
    index=movie_ids[:5],
))

Оценок: 1000209
Фильмов: 3706
Размерность матрицы жанров: (3706, 18)


,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
1,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Делим историю на Train, Validation и Test

Для каждого пользователя сортируем оценки по времени. Все взаимодействия, кроме двух последних, идут в Train. Предпоследний фильм оставляем для Validation, последний для Test.

Например, пользователь оценил фильмы A, B, C, D, E. Для предсказания D известны A, B, C. Для предсказания E уже известны A, B, C, D.

Так мы не используем будущие оценки при построении профиля пользователя.


In [83]:
# История каждого пользователя: от старой оценки к новой.
ratings = sort_interactions(ratings.copy())
ratings["movie_idx"] = ratings["movie_id"].map(movie2idx)
ratings["position"] = ratings.groupby("user_id").cumcount()
ratings["n_events"] = ratings.groupby("user_id")["movie_idx"].transform("size")

# Train , все оценки, кроме двух последних у каждого пользователя.
train_ratings = ratings.loc[
    ratings["position"] < ratings["n_events"] - 2,
    ["user_id", "movie_idx", "rating"],
].copy()

# Validation , предпоследняя оценка.
val_ratings = ratings.loc[
    ratings["position"] == ratings["n_events"] - 2,
    ["user_id", "movie_idx", "rating"],
].copy()

# Test , последняя оценка. Для предсказания нужен только movie_idx.
test_targets = ratings.loc[
    ratings["position"] == ratings["n_events"] - 1,
    ["user_id", "movie_idx"],
].copy()
val_targets = val_ratings[["user_id", "movie_idx"]].copy()

# Для Test уже известна и оценка валидационного фильма.
test_known_ratings = pd.concat(
    [train_ratings, val_ratings],
    ignore_index=True,
)

print("Train взаимодействий:", len(train_ratings))
print("Validation пользователей:", len(val_targets))
print("Test пользователей:", len(test_targets))
print("Известных оценок для Test:", len(test_known_ratings))

Train взаимодействий: 988129
Validation пользователей: 6040
Test пользователей: 6040
Известных оценок для Test: 994169


## 4. Считаем популярность и собираем истории пользователей

Популярность фильма считаем по числу оценок в Train среди всех пользователей. Она пригодится, если жанровый профиль оказался нулевым, и при одинаковой Cosine Similarity.

Также соберем отдельный словарь с историей каждого пользователя. Так не придется каждый раз искать его оценки в большой таблице.

Для Novelty используем только Train, чтобы оценка была сопоставима с другими моделями.


In [84]:
# Количество обучающих взаимодействий по каждому фильму.
popularity_scores = np.bincount(
    train_ratings["movie_idx"].to_numpy(),
    minlength=num_movies,
)

# Истории для построения профилей: отдельно для Validation и Test.
train_by_user = {
    int(user_id): group
    for user_id, group in train_ratings.groupby("user_id")
}
test_by_user = {
    int(user_id): group
    for user_id, group in test_known_ratings.groupby("user_id")
}

# Для расчёта Novelty внутри уже готовой metrics.py.
train_histories = (
    train_ratings.groupby("user_id")["movie_idx"]
    .apply(list)
    .to_dict()
)

assert len(train_by_user) == len(val_targets) == len(test_targets)
assert genre_matrix.shape[0] == num_movies

print("Пользователей:", len(train_by_user))
print("Популярность фильмов:", popularity_scores.shape)
print("История пользователя №1 (первые 5 индексов):", train_histories[1][:5])

Пользователей: 6040
Популярность фильмов: (3706,)
История пользователя №1 (первые 5 индексов): [2969, 1178, 1574, 957, 2147]


## 5. Посмотрим рекомендации одного пользователя

Для начала проверяем обычную Content-Based с `gamma=1.0`. Профиль строим по Train, после чего исключаем все фильмы из известной истории пользователя.

Таблица ниже сохраняет порядок Top-10.


In [85]:
user_id = 1

sample_top10 = cb.recommend_content_based(
    user_id=user_id,
    train_ratings=train_by_user[user_id],
    genre_matrix=genre_matrix,
    popularity_scores=popularity_scores,
    k=K,
    gamma=1.0,
)

# Превращаем movie_idx обратно в оригинальные movie_id.
recommended_movie_ids = [movie_ids[idx] for idx in sample_top10]

movie_info = movies.set_index("movie_id")
example_table = movie_info.loc[
    recommended_movie_ids, ["title", "genres"]
].copy()
example_table.index = range(1, len(example_table) + 1)
example_table.index.name = "rank"

display(example_table)

,title,genres
rank,,
1,Babe (1995),Children's|Comedy|Drama
2,Pollyanna (1960),Children's|Comedy|Drama
3,Wide Awake (1998),Children's|Comedy|Drama
4,Old Yeller (1957),Children's|Drama
5,"Parent Trap, The (1998)",Children's|Drama
6,"Parent Trap, The (1961)",Children's|Drama
7,"Little Princess, A (1995)",Children's|Drama
8,"Little Princess, The (1939)",Children's|Drama
9,Prancer (1989),Children's|Drama


## 6. Как работает временное затухание

Раньше мы использовали только вес оценки `rating - 3`. Теперь дополнительно умножаем его на `gamma` в степени, которая зависит от давности взаимодействия.

При `gamma=1` старые и новые оценки не отличаются по времени. При `gamma=0.6` новые оценки заметно сильнее влияют на профиль.

Проверим на четырех оценках, расположенных от старой к новой.


In [86]:
# Небольшой пример: 4 оценки от старой к новой.
ratings_example = np.array([5, 4, 2, 5])
gamma_example = 0.6

rating_weights = ratings_example - 3
n = len(rating_weights)
decay = gamma_example ** np.arange(n - 1, -1, -1)
temporal_weights = rating_weights * decay

weights_example = pd.DataFrame({
    "rating": ratings_example,
    "rating - 3": rating_weights,
    "decay": decay,
    "итоговый вес": temporal_weights,
}, index=["старый A", "B", "C", "новый D"])

display(weights_example.round(3))

,rating,rating - 3,decay,итоговый вес
старый A,5,2,0.216,0.432
B,4,1,0.360,0.360
C,2,-1,0.600,-0.600
новый D,5,2,1.000,2.000


## 7. Функция для рекомендаций всем пользователям

Для Validation и Test нужен одинаковый цикл, поэтому вынесем его в функцию `get_topk_for_users()`.

`targets_df` содержит пользователей и фильмы, которые мы пытаемся угадать. В `histories_by_user` находятся оценки, известные на момент предсказания. Параметр `gamma` отвечает за влияние старых оценок.

Функция возвращает рекомендации и правильные ответы в одном порядке. `tqdm` просто показывает прогресс обработки пользователей.


In [87]:
def get_topk_for_users(targets_df, histories_by_user, gamma, k=10):
    recommendations = []
    targets = []

    for row in tqdm(
        targets_df.itertuples(index=False),
        total=len(targets_df),
        desc=f"gamma={gamma}",
    ):
        user_id = int(row.user_id)
        target = int(row.movie_idx)

        # Известная история только этого пользователя.
        user_ratings = histories_by_user[user_id]

        # Рекомендуем фильмы, не входящие в его известную историю.
        topk = cb.recommend_content_based(
            user_id=user_id,
            train_ratings=user_ratings,
            genre_matrix=genre_matrix,
            popularity_scores=popularity_scores,
            k=k,
            gamma=gamma,
        )

        recommendations.append(topk)
        targets.append(target)

    return recommendations, targets

## 8. Считаем метрики

Все четыре метрики уже реализованы в `metrics.py`. Поэтому здесь достаточно вызвать готовые функции.

`HR@10` проверяет, попал ли нужный фильм в рекомендации. `NDCG@10` дополнительно учитывает его место. `Coverage@10` показывает долю фильмов каталога, попавших хотя бы в одну рекомендацию. `Novelty@10` оценивает, насколько редко рекомендуемые фильмы встречались в Train.


In [88]:
def evaluate_recommendations(recommendations, targets):
    return {
        "HR@10": metrics.hit_rate_at_k(recommendations, targets),
        "NDCG@10": metrics.ndcg_at_k(recommendations, targets),
        "Coverage@10": metrics.catalog_coverage_at_k(
            recommendations, num_movies
        ),
        "Novelty@10": metrics.novelty_at_k(
            recommendations, train_histories, num_movies
        ),
    }

## 9. Подбираем gamma на Validation

Проверим значения `gamma` от 1.0 до 0.5. Для каждого значения строим профиль только по Train и пытаемся угадать предпоследний фильм пользователя.

Основная метрика при выборе параметра у нас `NDCG@10`. На Test параметры не подбираем.


In [89]:
gammas = [1.0, 0.99, 0.98, 0.95, 0.90, 0.85, 0.80, 0.70, 0.60, 0.50]
val_rows = []

for gamma in gammas:
    # На Validation доступны только взаимодействия Train.
    recommendations, targets = get_topk_for_users(
        targets_df=val_targets,
        histories_by_user=train_by_user,
        gamma=gamma,
        k=K,
    )

    # Сохраняем значение gamma вместе с четырьмя метриками.
    scores = evaluate_recommendations(recommendations, targets)
    val_rows.append({"gamma": gamma, **scores})

val_comparison = pd.DataFrame(val_rows)

display(val_comparison.sort_values("NDCG@10", ascending=False).round(4))

gamma=0.5: 100%|██████████| 6040/6040 [00:04<00:00, 1291.44it/s]


,gamma,HR@10,NDCG@10,Coverage@10,Novelty@10
8,0.60,0.0445,0.0238,0.4603,11.0103
7,0.70,0.0447,0.0236,0.4574,11.0247
9,0.50,0.0439,0.0231,0.4603,11.0004
6,0.80,0.0434,0.0220,0.4517,11.0192
5,0.85,0.0421,0.0210,0.4485,11.0159
4,0.90,0.0389,0.0193,0.4490,11.0118
3,0.95,0.0348,0.0182,0.4482,11.0143
2,0.98,0.0298,0.0154,0.4514,11.0046
1,0.99,0.0257,0.0131,0.4404,10.9930
0,1.00,0.0243,0.0127,0.4317,10.9956


### Выбираем лучший коэффициент

Берем значение `gamma` с максимальным `NDCG@10` на Validation. В предыдущем эксперименте лучше всего показало себя `0.6`. Здесь получим результат заново из таблицы, а не будем подставлять его вручную.


In [90]:
best_row = val_comparison.loc[val_comparison["NDCG@10"].idxmax()]
best_gamma = float(best_row["gamma"])

print(f"Лучшая gamma на Validation: {best_gamma:.2f}")
print(f"HR@10:   {best_row['HR@10']:.4f}")
print(f"NDCG@10: {best_row['NDCG@10']:.4f}")

Лучшая gamma на Validation: 0.60
HR@10:   0.0445
NDCG@10: 0.0238


## 10. Проверяем на Test

Теперь для каждого пользователя известны Train и его валидационная оценка. Используем их, чтобы построить профиль, и пытаемся угадать последний фильм.

Посчитаем метрики для двух вариантов: обычной Content-Based с `gamma=1.0` и Temporal Content-Based с лучшим `gamma`, выбранным на Validation.


In [91]:
test_rows = []

for model_name, gamma in [
    ("Content-Based", 1.0),
    ("Temporal Content-Based", best_gamma),
]:
    recommendations, targets = get_topk_for_users(
        targets_df=test_targets,
        histories_by_user=test_by_user,
        gamma=gamma,
        k=K,
    )

    scores = evaluate_recommendations(recommendations, targets)
    test_rows.append({"Model": model_name, **scores})

test_comparison = pd.DataFrame(test_rows).set_index("Model")
display(test_comparison.round(4))

gamma=0.6: 100%|██████████| 6040/6040 [00:04<00:00, 1287.73it/s]


,HR@10,NDCG@10,Coverage@10,Novelty@10
Model,,,,
Content-Based,0.0262,0.0135,0.4328,11.0040
Temporal Content-Based,0.0396,0.0218,0.4595,11.0235


## 11. Сравниваем модели

Результаты Transformer и Popularity берем из `02_PyTorch_Basics.ipynb`. В этом ноутбуке они не пересчитываются. Результаты двух Content-Based моделей только что получили на Test.


In [92]:
previous_results = pd.DataFrame(
    {
        "HR@10": [0.2954, 0.0369],
        "NDCG@10": [0.1714, 0.0180],
        "Coverage@10": [0.6638, 0.0537],
        "Novelty@10": [10.7081, 8.6419],
    },
    index=["Transformer", "Popularity"],
)

final_comparison = pd.concat([previous_results, test_comparison])
display(final_comparison.round(4))

,HR@10,NDCG@10,Coverage@10,Novelty@10
Transformer,0.2954,0.1714,0.6638,10.7081
Popularity,0.0369,0.0180,0.0537,8.6419
Content-Based,0.0262,0.0135,0.4328,11.0040
Temporal Content-Based,0.0396,0.0218,0.4595,11.0235


## 12. Что получилось

Temporal Content-Based учитывает недавние оценки сильнее старых. Подбор `gamma` делали на Validation, а выбранное значение проверили на Test.

У жанровой модели есть ограничения. Даже с учетом времени она плохо различает фильмы с одинаковыми жанрами, поскольку больше информации об объектах у нее нет. В следующем эксперименте попробуем использовать дополнительные признаки и перейти к Feature-Based / Hybrid Recommender.

Мы уже смотрели на Test при разработке базовой модели, поэтому это пока исследовательское сравнение. Для независимой финальной оценки понадобится отдельный набор данных, который не использовался при выборе решений.
